# Класифікація зображень: Intel Image Classification

Ноутбук для запуску в Google Colab (**Runtime -> Change runtime type -> GPU**).

Порівнюємо два підходи:
- **Варіант A:** проста CNN, написана з нуля;
- **Варіант B:** transfer learning (ResNet18, попередньо навчена на ImageNet).

Аналіз результатів, критика архітектури та шляхи покращення описані текстом у кінці ноутбука.

## 1. Імпорт бібліотек

In [ ]:
!pip install -q kagglehub

import copy
import random
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Завантаження та підготовка даних

Датасет уже розділений на `seg_train` і `seg_test`. З тренувальної частини відділяємо 20% для валідації (стратифіковано, щоб пропорції класів збереглися).

In [ ]:
dataset_path = Path(kagglehub.dataset_download('puneet6060/intel-image-classification'))
train_root = dataset_path / 'seg_train' / 'seg_train'
test_root = dataset_path / 'seg_test' / 'seg_test'
print('Dataset:', dataset_path)
print('train dir exists:', train_root.is_dir(), '| test dir exists:', test_root.is_dir())

In [ ]:
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
MEAN = (0.485, 0.456, 0.406)  # статистика ImageNet, на якій навчена ResNet18
STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_train = datasets.ImageFolder(train_root, transform=transform)
test_set = datasets.ImageFolder(test_root, transform=transform)
classes = full_train.classes

train_idx, val_idx = train_test_split(
    np.arange(len(full_train)),
    test_size=0.2,
    stratify=full_train.targets,
    random_state=SEED,
)
train_set = Subset(full_train, train_idx)
val_set = Subset(full_train, val_idx)

loaders = {
    name: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(name == 'train'), num_workers=2)
    for name, ds in (('train', train_set), ('val', val_set), ('test', test_set))
}

print('Classes:', classes)
for name, loader in loaders.items():
    print(f'{name}: {len(loader.dataset)} images')

## 3. Моделі

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        # 150x150 -> 18x18 після трьох MaxPool2d(2)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 18 * 18, 256), nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def create_transfer_model(num_classes):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for parameter in model.parameters():
        parameter.requires_grad = False  # заморожуємо попередньо навчений екстрактор ознак
    model.fc = nn.Linear(model.fc.in_features, num_classes)  # навчається лише новий класифікатор
    return model


models = {
    'simple_cnn': SimpleCNN(len(classes)),
    'resnet18': create_transfer_model(len(classes)),
}
for name, model in models.items():
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'{name}: trainable params = {trainable:,} / {total:,}')

## 4. Функція втрат, оптимізатор і гіперпараметри

- **CrossEntropyLoss:** задача багатокласова з взаємовиключними класами; функція приймає логіти й індекси класів, softmax додавати не потрібно.
- **Adam, lr = 1e-3:** адаптивний крок для кожного параметра, надійний базовий вибір, швидко збігається без ручного підбору momentum. Для ResNet18 оптимізуються лише параметри, що не заморожені.
- **batch size = 32:** компроміс між стабільністю градієнта та обсягом пам'яті GPU; помірний шум градієнта також діє як легка регуляризація.
- **10 епох:** на графіках нижче видно, чи виходять криві на плато; найкращі ваги вибираються за мінімумом validation loss, тому надлишок епох не шкодить результату.
- **Метрики:** accuracy і **macro F1**. Класи майже збалансовані, тож accuracy інформативна, а macro F1 додатково враховує precision і recall кожного класу з однаковою вагою.

## 5. Навчання

Для кожної епохи рахуємо loss і accuracy на train та val. Зберігаємо ваги з найменшим validation loss.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct = 0.0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
    size = len(loader.dataset)
    return total_loss / size, correct / size


def train_model(model, name, num_epochs=10, lr=1e-3):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss, best_state = float('inf'), None

    for epoch in range(num_epochs):
        train_loss, train_acc = run_epoch(model, loaders['train'], criterion, optimizer)
        val_loss, val_acc = run_epoch(model, loaders['val'], criterion)
        for key, value in zip(history, (train_loss, val_loss, train_acc, val_acc)):
            history[key].append(value)
        if val_loss < best_val_loss:
            best_val_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
            torch.save(best_state, f'best_{name}.pt')
        print(
            f'[{name}] epoch {epoch + 1}/{num_epochs}: '
            f'train_loss={train_loss:.4f} val_loss={val_loss:.4f} '
            f'train_acc={train_acc:.4f} val_acc={val_acc:.4f}'
        )

    model.load_state_dict(best_state)  # далі використовуємо найкращу за val loss модель
    return history


histories = {name: train_model(model, name) for name, model in models.items()}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, (name, h) in enumerate(histories.items()):
    axes[row, 0].plot(h['train_loss'], label='train')
    axes[row, 0].plot(h['val_loss'], label='validation')
    axes[row, 0].set_title(f'{name}: loss')
    axes[row, 1].plot(h['train_acc'], label='train')
    axes[row, 1].plot(h['val_acc'], label='validation')
    axes[row, 1].set_title(f'{name}: accuracy')
for ax in axes.ravel():
    ax.set_xlabel('Epoch')
    ax.legend()
fig.tight_layout()
plt.show()

## 6. Оцінка моделей на train, validation і test

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    predictions, labels = [], []
    for images, y in loader:
        predictions += model(images.to(device)).argmax(1).cpu().tolist()
        labels += y.tolist()
    return np.array(labels), np.array(predictions)


results, rows = {}, []
for name, model in models.items():
    for split, loader in loaders.items():
        y, p = predict(model, loader)
        results[(name, split)] = (y, p)
        rows.append({
            'model': name,
            'split': split,
            'accuracy': accuracy_score(y, p),
            'macro_f1': f1_score(y, p, average='macro'),
            'weighted_f1': f1_score(y, p, average='weighted'),
        })

metrics_table = pd.DataFrame(rows)
print(metrics_table.round(4).to_string(index=False))

In [ ]:
for name in models:
    y, p = results[(name, 'test')]
    print(f'=== {name}: test classification report ===')
    print(classification_report(y, p, target_names=classes, digits=3))

    m = confusion_matrix(y, p)
    confusions = sorted(
        ((int(m[i, j]), classes[i], classes[j]) for i in range(len(classes)) for j in range(len(classes)) if i != j),
        reverse=True,
    )[:5]
    print('Top-5 confusions (count, true -> predicted):')
    for count, true_name, pred_name in confusions:
        print(f'  {count:4d}  {true_name} -> {pred_name}')
    print()

## 7. Візуалізація фактичних і прогнозованих класів

Матриці помилок для train, validation і test.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(21, 13))
for row, name in enumerate(models):
    for col, split in enumerate(loaders):
        y, p = results[(name, split)]
        m = confusion_matrix(y, p)
        ax = axes[row, col]
        ax.imshow(m, cmap='Blues')
        ax.set(
            title=f'{name} - {split} (acc={accuracy_score(y, p):.3f})',
            xlabel='Predicted class',
            ylabel='True class',
            xticks=range(len(classes)),
            yticks=range(len(classes)),
            xticklabels=classes,
            yticklabels=classes,
        )
        for r in range(len(classes)):
            for c in range(len(classes)):
                ax.text(c, r, m[r, c], ha='center', va='center')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
fig.tight_layout()
plt.show()

Приклади тестових зображень: 8 помилкових (червоний підпис) і 8 правильних (зелений).

In [ ]:
def denormalize(image):
    return (image.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)).clip(0, 1)


rng = np.random.default_rng(SEED)
for name in models:
    y, p = results[(name, 'test')]
    wrong = np.where(y != p)[0]
    right = np.where(y == p)[0]
    chosen = list(rng.choice(wrong, 8, replace=False)) + list(rng.choice(right, 8, replace=False))

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for ax, i in zip(axes.ravel(), chosen):
        image, _ = test_set[i]
        ax.imshow(denormalize(image))
        ax.set_title(
            f'true: {classes[y[i]]}\npred: {classes[p[i]]}',
            fontsize=9,
            color='green' if y[i] == p[i] else 'red',
        )
        ax.axis('off')
    fig.suptitle(f'{name}: test samples (actual vs predicted)')
    fig.tight_layout()
    plt.show()

## 8. Висновки

У мене вийшло приблизно таке (при повторному запуску цифри можуть трохи відрізнятися):

| Модель | Train accuracy | Val accuracy | Test accuracy |
|---|---|---|---|
| SimpleCNN | ~0.89 | ~0.81 | ~0.80 |
| ResNet18 (transfer learning) | ~0.91 | ~0.90 | ~0.90 |

### 1. Аналіз результатів

**Чому результати такі.**

- **SimpleCNN** показала ~80% на тесті, це в межах очікуваних 75-85%. Але на train точність ~89%, а на test ~80%. Така різниця означає, що модель **перенавчилась**: вона трохи запам'ятала тренувальні картинки, а на нових працює гірше. Я не використовував аугментацію, Dropout і BatchNorm, тому нічого не заважало їй запам'ятовувати.
- **ResNet18** показала ~89.5%, тобто трохи менше за очікувані 90-97%. Тут різниця між train і test маленька (~2%), тож перенавчання немає. Скоріше модель **недонавчена**: я заморозив усі шари, крім останнього, і навчалась тільки вона. Ознаки з ImageNet добрі, але не ідеально підходять саме для цих сцен.
- Я також не підбирав `lr`, batch size і кількість епох окремо (взяла стандартні значення), тож результат можна було б покращити.

**Які помилки робить модель.**

- Найкраще розпізнається `forest` (F1 ~0.93 у SimpleCNN і ~0.98 у ResNet18), бо ліс виглядає дуже характерно.
- Найчастіше модель плутає схожі класи: `glacier` і `mountain` (гори зі снігом), `buildings` і `street` (будинки на вулиці). SimpleCNN ще іноді плутає `sea` з `mountain`.
- Результати на val і test майже однакові, тому валідаційний набір добре показує, як модель працюватиме на нових даних.

**Обмеження даних і підходу.**

- Деякі картинки містять одразу кілька речей (наприклад, гори й озеро), а мітка тільки одна. Тому навіть людина інколи не знала б, яка відповідь правильна, і 100% точності досягти неможливо.
- У датасеті можуть бути неправильно підписані картинки.
- Картинки маленькі (150x150), дрібні деталі (сніг, вікна) погано видно, а саме вони відрізняють схожі класи.
- Я зробив лише один поділ на train/val і один запуск навчання. Тому різниця в 1-2% між результатами може бути просто випадковістю.

### 2. Оцінка архітектури

**Чому я вибрав таку архітектуру.**

- **SimpleCNN:** три блоки `Conv2d -> ReLU -> MaxPool2d`, кількість фільтрів росте 32 -> 64 -> 128. Перші шари знаходять прості речі (лінії, кольори), наступні складають з них складніші форми. Це стандартна і проста схема, яка швидко навчається, тож добре підходить як базовий варіант для порівняння. В кінці два повнозв'язні шари дають 6 оцінок для 6 класів.
- **ResNet18 із замороженими шарами:** це найпростіший спосіб transfer learning. Модель вже навчена на мільйонах картинок ImageNet, тому вона вже вміє знаходити корисні ознаки. Я замінив тільки останній шар під 6 класів, тож навчати треба дуже мало параметрів, і ризик перенавчання менший.

**Недоліки SimpleCNN.**

- Немає BatchNorm, тому навчання менш стабільне.
- Немає Dropout і weight decay, тому нічого не стримує перенавчання.
- Перший повнозв'язний шар дуже великий (`41472 -> 256`, понад 10 млн параметрів), і саме він найбільше запам'ятовує дані.
- Всього 3 згорткові шари, цього може бути мало, щоб відрізнити схожі сцени.
- Немає аугментації, тому модель чутлива до змін освітлення чи положення об'єктів.

**Недоліки ResNet18 у моєму варіанті.**

- Я заморозив майже всю модель, і вона не може підлаштуватися під наші картинки. Мабуть, через це результат ~89.5%, а не 90%+.
- ResNet навчалась на картинках 224x224, а в мене 150x150, тому розмір об'єктів трохи інший.
- ResNet18 порівняно невелика модель, їй може не вистачати можливостей для схожих класів.

### 3. Як можна покращити

**Що можна зробити з поточними моделями.**

1. **Додати аугментацію** для train: випадкове віддзеркалення, обрізання, зміна яскравості. Це найпростіше і має допомогти SimpleCNN проти перенавчання.
2. **Покращити SimpleCNN:** додати BatchNorm, Dropout, weight decay, а замість `Flatten` і великого шару використати `AdaptiveAvgPool2d`, щоб було менше параметрів.
3. **Fine-tuning ResNet18:** розморозити останній блок (`layer4`) і навчати його з малим `lr` (~1e-4). Скоріше за все це дасть найбільший приріст, до 92-95%.
4. **Додати scheduler** (зменшення `lr` під час навчання) і early stopping (зупинка, коли val loss більше не падає).
5. **Більше епох** (20-30) разом з early stopping.
6. **Підібрати гіперпараметри** (`lr`, batch size, weight decay) і запустити навчання кілька разів з різними seed, щоб бачити, наскільки результат стабільний.
7. **Збільшити розмір картинок** до 224x224 для ResNet.
8. **Label smoothing** може допомогти, бо частина міток неоднозначна.
9. **Подивитися на найбільш хибні прогнози** і перевірити, чи не підписані ці картинки неправильно.

**Інші підходи, які можуть дати кращий результат.**

- **Потужніші моделі:** ResNet50, EfficientNet, Vision Transformer. Зазвичай на таких задачах вони дають 94-97%.
- **Ансамбль:** усереднити відповіді кількох моделей, це часто додає 1-2%.
- **Test-Time Augmentation:** зробити кілька змінених копій однієї картинки і усереднити прогноз.
- **MixUp / CutMix:** змішування картинок під час навчання, допомагає проти перенавчання.
- **Двоетапне навчання:** спочатку навчити тільки останній шар, потім розморозити решту з дуже малим `lr`.
- **Multi-label підхід:** оскільки на багатьох фото є кілька об'єктів, можна дозволити моделі вказувати кілька класів.
- **Більше даних:** використати `seg_pred` (картинки без міток) для pseudo-labeling або додаткові датасети сцен, наприклад Places365.

### Загальний підсумок

SimpleCNN працює як нормальна базова модель (~80%), але перенавчається, бо в ній немає регуляризації й аугментації. ResNet18 з transfer learning працює краще (~89.5%) і стабільніше, хоча я навчав лише один шар. Далі результат можна покращити через fine-tuning, аугментацію і потужніші моделі. Більшість помилок спричинені схожими класами і неоднозначними мітками, а не лише слабкою моделлю, тому 100% тут навряд чи можливі.